The LLM is allowed to propose a decision, but it is not allowed to directly execute the action. The workflow pauses for human approval, and only after approval does the application execute the action

```
User Request
      ↓
     LLM
      ↓
Structured Proposed Action

{
  action: "refund",
  customer: "CUST-101",
  amount: 5000,
  reason: "Customer received a damaged product"
}

      ↓
 HUMAN REVIEW
      ↓

Approve? yes / no

     /      \
   yes       no
    ↓         ↓
 Execute     Stop
 Refund

 ```

In [ ]:
# pip install langchain-openai pydantic

from typing import Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI


# ============================================================
# LLM
# ============================================================

model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)


# ============================================================
# STRUCTURED OUTPUT
# ============================================================

class ProposedAction(BaseModel):

    action: Literal[
        "refund",
        "reject",
        "manual_review"
    ]

    customer: str = Field(
        description="Customer ID"
    )

    amount: float = Field(
        description="Refund amount"
    )

    reason: str = Field(
        description="Reason for the proposed action"
    )


structured_model = model.with_structured_output(
    ProposedAction
)


# ============================================================
# AI AGENT
# ============================================================

def ai_agent(user_request):

    prompt = f"""
You are a customer support AI agent.

Analyze the following customer request:

{user_request}

Decide the appropriate action.

Possible actions:
- refund
- reject
- manual_review

Return:
- action
- customer ID
- refund amount
- reason

Do not execute the action.
Only propose the action for human approval.
"""

    proposed_action = structured_model.invoke(
        prompt
    )

    return proposed_action


# ============================================================
# HUMAN REVIEW
# ============================================================

def human_review(action):

    print("\n========== AI Proposed Action ==========")

    print("Action   :", action.action)
    print("Customer :", action.customer)
    print("Amount   :", action.amount)
    print("Reason   :", action.reason)

    print("========================================")


    decision = input(
        "\nApprove this action? (yes/no): "
    ).strip().lower()


    if decision == "yes":
        return True

    return False


# ============================================================
# EXECUTE ACTION
# ============================================================

def execute_action(action):

    if action.action == "refund":

        print(
            f"\nRefund of ₹{action.amount} "
            f"processed for {action.customer}."
        )


    elif action.action == "reject":

        print(
            f"\nRequest rejected for "
            f"{action.customer}."
        )


    elif action.action == "manual_review":

        print(
            f"\nRequest for {action.customer} "
            f"sent for manual review."
        )


# ============================================================
# MAIN WORKFLOW
# ============================================================

def run():

    user_request = input(
        "User Request: "
    )


    # Step 1
    # LLM analyzes request and proposes action

    proposed_action = ai_agent(
        user_request
    )


    # Step 2
    # Human reviews the AI decision

    approved = human_review(
        proposed_action
    )


    # Step 3
    # Execute only after human approval

    if approved:

        execute_action(
            proposed_action
        )

    else:

        print(
            "\nAction rejected by human."
        )


# ============================================================
# RUN
# ============================================================

run()

## Using Langgraph

              
              
                ┌─────────────┐
User Request →  │ LLM Analyze │
                └──────┬──────┘
                       ↓
               Proposed Action
                       ↓
                ┌───────────┐
                │ interrupt │
                └─────┬─────┘
                      │
                 GRAPH PAUSED
                      │
                      ↓
               Human Approval
                /           \
             Yes             No
              │               │
              ↓               ↓
Command(resume=True)   Command(resume=False)
              │               │
              ↓               ↓
           Execute           Reject

In [ ]:
# pip install -U langgraph langchain-openai pydantic


from typing import TypedDict, Literal

from pydantic import BaseModel

from langchain_openai import ChatOpenAI

from langgraph.graph import (
    StateGraph,
    START,
    END
)

from langgraph.checkpoint.memory import (
    InMemorySaver
)

from langgraph.types import (
    interrupt,
    Command
)


# ============================================================
# LLM
# ============================================================

model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)


# ============================================================
# STRUCTURED OUTPUT FOR LLM
# ============================================================

class ProposedAction(BaseModel):

    action: Literal[
        "refund",
        "reject",
        "manual_review"
    ]

    customer: str

    amount: float

    reason: str


structured_model = model.with_structured_output(
    ProposedAction
)


# ============================================================
# GRAPH STATE
# ============================================================

class AgentState(TypedDict):

    user_request: str

    action: str

    customer: str

    amount: float

    reason: str

    approved: bool

    final_message: str


# ============================================================
# NODE 1
# LLM PROPOSES ACTION
# ============================================================

def analyze_request(
    state: AgentState
):

    prompt = f"""
You are a customer support AI agent.

Analyze the following customer request:

{state["user_request"]}

Decide the appropriate action.

Possible actions:
- refund
- reject
- manual_review

Return:
- action
- customer
- amount
- reason

Do not execute anything.
Only propose an action.
"""


    result = structured_model.invoke(
        prompt
    )


    return {
        "action": result.action,
        "customer": result.customer,
        "amount": result.amount,
        "reason": result.reason
    }


# ============================================================
# NODE 2
# HUMAN APPROVAL
# ============================================================

def human_approval(
    state: AgentState
):

    # Graph pauses here

    decision = interrupt(
        {
            "message": "Please review the AI proposed action.",

            "action": state["action"],

            "customer": state["customer"],

            "amount": state["amount"],

            "reason": state["reason"]
        }
    )


    # decision comes from Command(resume=...)
    return {
        "approved": decision
    }


# ============================================================
# ROUTING FUNCTION
# ============================================================

def approval_router(
    state: AgentState
):

    if state["approved"]:
        return "execute"

    return "reject"


# ============================================================
# NODE 3A
# EXECUTE APPROVED ACTION
# ============================================================

def execute_action(
    state: AgentState
):

    if state["action"] == "refund":

        message = (
            f"Refund of ₹{state['amount']} "
            f"processed for "
            f"{state['customer']}."
        )


    elif state["action"] == "reject":

        message = (
            f"Request rejected for "
            f"{state['customer']}."
        )


    else:

        message = (
            f"Request for "
            f"{state['customer']} "
            f"sent for manual review."
        )


    return {
        "final_message": message
    }


# ============================================================
# NODE 3B
# HUMAN REJECTED ACTION
# ============================================================

def reject_action(
    state: AgentState
):

    return {
        "final_message":
        "Action rejected by human reviewer."
    }


# ============================================================
# BUILD GRAPH
# ============================================================

builder = StateGraph(
    AgentState
)


builder.add_node(
    "analyze",
    analyze_request
)


builder.add_node(
    "human_approval",
    human_approval
)


builder.add_node(
    "execute",
    execute_action
)


builder.add_node(
    "reject",
    reject_action
)


# ============================================================
# EDGES
# ============================================================

builder.add_edge(
    START,
    "analyze"
)


builder.add_edge(
    "analyze",
    "human_approval"
)


builder.add_conditional_edges(
    "human_approval",
    approval_router,
    {
        "execute": "execute",
        "reject": "reject"
    }
)


builder.add_edge(
    "execute",
    END
)


builder.add_edge(
    "reject",
    END
)


# ============================================================
# CHECKPOINTER
# ============================================================

checkpointer = InMemorySaver()


# ============================================================
# COMPILE GRAPH
# ============================================================

graph = builder.compile(
    checkpointer=checkpointer
)


# ============================================================
# THREAD CONFIG
# ============================================================

config = {
    "configurable": {
        "thread_id": "refund_thread_1"
    }
}


# ============================================================
# STEP 1
# START GRAPH
# ============================================================

result = graph.invoke(

    {
        "user_request":
        """
Customer CUST-101 received a damaged product.
Please refund ₹5000.
""",

        "action": "",
        "customer": "",
        "amount": 0,
        "reason": "",
        "approved": False,
        "final_message": ""
    },

    config=config
)


# ============================================================
# GRAPH WILL PAUSE AT interrupt()
# ============================================================

print(
    "\nGraph paused for human approval."
)


print(
    "\nInterrupt Information:"
)


print(
    result["__interrupt__"]
)


# ============================================================
# GET HUMAN DECISION
# ============================================================

human_input = input(
    "\nApprove action? (yes/no): "
).strip().lower()


approved = (
    human_input == "yes"
)


# ============================================================
# STEP 2
# RESUME GRAPH
# ============================================================

final_result = graph.invoke(

    Command(
        resume=approved
    ),

    config=config
)


# ============================================================
# FINAL RESULT
# ============================================================

print(
    "\nFinal Result:"
)

print(
    final_result["final_message"]
)